# **Métricas modelos predictivos**

En este notebook, recuperamos los .csv de las predicciones de los modelos, miramos cuál fue la mejor época según el [MCC](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.matthews_corrcoef.html) ,sacamos métricas de los modelos y las exportamos a csv.

Por otro lado, también sacamos las métricas del modelo destilado.

In [ ]:
from huggingface_hub import download_bucket_files
from pathlib import Path
import os
from huggingface_hub import HfApi, list_bucket_tree
import pandas as pd
import os
from sklearn.metrics import accuracy_score, matthews_corrcoef, f1_score, recall_score

In [ ]:
os.environ["HF_TOKEN"] = "mi_token"

bucket_id = "z72pepee/tfm-explicabilidad"
remote_path = "Modelos_Entrenados"

token= "mi_token"
api = HfApi(token=token)


download_bucket_files(
    bucket_id,
    files=[
        ("Modelos_Entrenados/experimento_dataset1_microsoft/test.csv", "./test_modelo_1.csv"),
        ("Modelos_Entrenados/experimento_dataset1_microsoft/val.csv", "./val_modelo_1.csv"),
        ("Modelos_Entrenados/experimento_dataset2_microsoft/val.csv", "./val_modelo_2.csv"),
        ("Modelos_Entrenados/experimento_dataset2_microsoft/test.csv", "./test_modelo_2.csv"),
        ("Modelos_Entrenados/experimento_dataset3_microsoft/test.csv", "./test_modelo_3.csv"),
        ("Modelos_Entrenados/experimento_dataset3_microsoft/val.csv", "./val_modelo_3.csv"),
        ("Modelos_Entrenados/experimento_dataset1_googleViT/test.csv", "./test_modelo_4.csv"),
        ("Modelos_Entrenados/experimento_dataset1_googleViT/val.csv", "./val_modelo_4.csv"),
        ("Modelos_Entrenados/experimento_dataset2_googleViT/val.csv", "./val_modelo_5.csv"),
        ("Modelos_Entrenados/experimento_dataset2_googleViT/test.csv", "./test_modelo_5.csv"),
        ("Modelos_Entrenados/experimento_dataset3_googleViT/val.csv", "./val_modelo_6.csv"),
        ("Modelos_Entrenados/experimento_dataset3_googleViT/test.csv", "./test_modelo_6.csv"),
        ("Modelos_Entrenados/experimento_dataset1_facebook_dinov2/test.csv", "./test_modelo_7.csv"),
        ("Modelos_Entrenados/experimento_dataset1_facebook_dinov2/val.csv", "./val_modelo_7.csv"),
        ("Modelos_Entrenados/experimento_dataset2_facebook_dinov2/val.csv", "./val_modelo_8.csv"),
        ("Modelos_Entrenados/experimento_dataset2_facebook_dinov2/test.csv", "./test_modelo_8.csv"),
        ("Modelos_Entrenados/experimento_dataset3_facebook_dinov2/val.csv", "./val_modelo_9.csv"),
        ("Modelos_Entrenados/experimento_dataset3_facebook_dinov2/test.csv", "./test_modelo_9.csv")
    ]
)

In [ ]:

# ==========================================
# CONFIGURACIÓN DE LOS DATASETS
# ==========================================
diccionario_dataset_1 = {
    0: '0_ACK', 1: '1_BCC', 2: '2_MEL',
    3: '3_NEV', 4: '4_SCC', 5: '5_SEK'
}

diccionario_dataset_2 = {
    0: '000_benign_dermal',
    1: '001_benign_epidermal',
    2: '002_benign_melanocyte',
    3: '003_genodermatoses',
    4: '004_inflammatory',
    5: '005_malignant_cutaneous_lymphoma',
    6: '006_malignant_dermal',
    7: '007_malignant_epidermal',
    8: '008_malignant_melanoma'
}

diccionario_dataset_3 = {
    0:'benign',
    1:'malignant'
    }

COL_GROUND_TRUTH = 'etiqueta_real' # Adaptado a nuestros .csv


In [ ]:
# ==========================================
# EL MOTOR DE EVALUACIÓN
# ==========================================
def buscar_mejor_epoch_y_reportar(df_val, df_test, nombre_modelo, mapa_clases):
    mejor_mcc = -1
    mejor_columna_epoch = None

    y_true_val = df_val[COL_GROUND_TRUTH]
    y_true_test = df_test[COL_GROUND_TRUTH]

    # Buscar automáticamente todas las columnas que sean epochs
    columnas_epochs = [col for col in df_val.columns if col.startswith('epoca_')]

    if not columnas_epochs:
        print(f" Error: No se encontraron columnas que empiecen por 'epoca_' en {nombre_modelo}")
        return

    # Iterar solo por las columnas detectadas automáticamente
    for col_epoch in columnas_epochs:
        # Calculamos el MCC para esta columna/epoch en concreto
        mcc_actual = matthews_corrcoef(y_true_val, df_val[col_epoch])

        # Si superamos el récord, guardamos el MCC y el NOMBRE de la columna
        if mcc_actual > mejor_mcc:
            mejor_mcc = mcc_actual
            mejor_columna_epoch = col_epoch

    # Extraer las predicciones ganadoras (usando el nombre de la mejor columna)
    y_pred_val_best = df_val[mejor_columna_epoch]
    y_pred_test_best = df_test[mejor_columna_epoch]

    # Calcular métricas finales
    acc_val = accuracy_score(y_true_val, y_pred_val_best)
    acc_test = accuracy_score(y_true_test, y_pred_test_best)
    mcc_test = matthews_corrcoef(y_true_test, y_pred_test_best)

    f1_mac_val = f1_score(y_true_val, y_pred_val_best, average='macro')
    f1_mac_test = f1_score(y_true_test, y_pred_test_best, average='macro')

    f1_wei_val = f1_score(y_true_val, y_pred_val_best, average='weighted')
    f1_wei_test = f1_score(y_true_test, y_pred_test_best, average='weighted')

    recalls_val = recall_score(y_true_val, y_pred_val_best, average=None)
    recalls_test = recall_score(y_true_test, y_pred_test_best, average=None)

    # Imprimir el Reporte
    print("\n" + "="*70)
    print(f" REPORTE DEL {nombre_modelo.upper()} | MEJOR: {mejor_columna_epoch.upper()}")
    print("="*70)
    print(f"{'Métrica':<18} | {'Validación':<15} | {'Test'}")
    print("-" * 70)
    print(f"{'Accuracy':<18} | {acc_val:.4f}          | {acc_test:.4f}")
    print(f"{'MCC':<18} | {mejor_mcc:.4f}          | {mcc_test:.4f}")
    print(f"{'F1 Macro':<18} | {f1_mac_val:.4f}          | {f1_mac_test:.4f}")
    print(f"{'F1 Weighted':<18} | {f1_wei_val:.4f}          | {f1_wei_test:.4f}")
    print("-" * 70)
    print("Recall por clase:")
    print("-" * 70)

    for clase_num, nombre in mapa_clases.items():
        if clase_num < len(recalls_val):
            r_val = recalls_val[clase_num]
            r_test = recalls_test[clase_num]
            print(f"  • {nombre:<14} | {r_val:.4f} ({r_val*100:.1f}%) | {r_test:.4f} ({r_test*100:.1f}%)")
    print("="*70 + "\n")


In [ ]:
# ==========================================
# EL BUCLE
# ==========================================
NUMERO_DE_MODELOS = 9
directorio_csvs = '' # Vacío si el script y los csv están juntos

print(" INICIANDO EVALUACIÓN  DE MODELOS...")

for i in range(1, NUMERO_DE_MODELOS + 1):
    archivo_val = os.path.join(directorio_csvs, f'val_modelo_{i}.csv')
    archivo_test = os.path.join(directorio_csvs, f'test_modelo_{i}.csv')

    if os.path.exists(archivo_val) and os.path.exists(archivo_test):
        df_validacion = pd.read_csv(archivo_val)
        df_testeo = pd.read_csv(archivo_test)
        if i % 3 ==1:
          MAPA_CLASES_ACTUAL= diccionario_dataset_1
        elif i % 3 == 2:
          MAPA_CLASES_ACTUAL= diccionario_dataset_2
        elif i % 3 == 0:
          MAPA_CLASES_ACTUAL= diccionario_dataset_3

        if i in range(1,4):
          print(" \n MODELO MICROSOFT")
        elif i in range(4,7):
          print ("\n MODELO GOOGLE ViT")
        elif i in range(7,10):
          print("\n MODELO FACEBOOK DINOV2")
        buscar_mejor_epoch_y_reportar(
            df_val=df_validacion,
            df_test=df_testeo,
            nombre_modelo=f"Modelo {i}",
            mapa_clases=MAPA_CLASES_ACTUAL
        )
    else:
        print(f" Faltan archivos para el Modelo {i}. Saltando...")

 INICIANDO EVALUACIÓN  DE MODELOS...
 
 MODELO MICROSOFT

 REPORTE DEL MODELO 1 | MEJOR: EPOCA_8
Métrica            | Validación      | Test
----------------------------------------------------------------------
Accuracy           | 0.8000          | 0.7478
MCC                | 0.7222          | 0.6609
F1 Macro           | 0.7992          | 0.6829
F1 Weighted        | 0.7969          | 0.7391
----------------------------------------------------------------------
Recall por clase:
----------------------------------------------------------------------
  • 0_ACK          | 0.8788 (87.9%) | 0.8462 (84.6%)
  • 1_BCC          | 0.8125 (81.2%) | 0.7568 (75.7%)
  • 2_MEL          | 1.0000 (100.0%) | 0.5000 (50.0%)
  • 3_NEV          | 0.6667 (66.7%) | 0.3000 (30.0%)
  • 4_SCC          | 0.5000 (50.0%) | 0.6000 (60.0%)
  • 5_SEK          | 1.0000 (100.0%) | 0.8824 (88.2%)

 
 MODELO MICROSOFT

 REPORTE DEL MODELO 2 | MEJOR: EPOCA_14
Métrica            | Validación      | Test
------------------

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, matthews_corrcoef, f1_score, recall_score

# ==========================================
# EL MOTOR DE EVALUACIÓN (Modificado para CSVs)
# ==========================================
def buscar_mejor_epoch_y_reportar(df_val, df_test, nombre_modelo, mapa_clases,ruta_salida):
    mejor_mcc = -1
    mejor_columna_epoch = None

    y_true_val = df_val[COL_GROUND_TRUTH]
    y_true_test = df_test[COL_GROUND_TRUTH]

    # Buscar automáticamente todas las columnas que sean epochs
    columnas_epochs = [col for col in df_val.columns if col.startswith('epoca_')]

    if not columnas_epochs:
        print(f" Error: No se encontraron columnas que empiecen por 'epoca_' en {nombre_modelo}")
        return

    # Iterar solo por las columnas detectadas automáticamente
    for col_epoch in columnas_epochs:
        # Calculamos el MCC para esta columna/epoch en concreto
        mcc_actual = matthews_corrcoef(y_true_val, df_val[col_epoch])

        # Si superamos el récord, guardamos el MCC y el NOMBRE de la columna
        if mcc_actual > mejor_mcc:
            mejor_mcc = mcc_actual
            mejor_columna_epoch = col_epoch

    # Extraer las predicciones ganadoras (usando el nombre de la mejor columna)
    y_pred_val_best = df_val[mejor_columna_epoch]
    y_pred_test_best = df_test[mejor_columna_epoch]

    # Calcular métricas finales
    acc_val = accuracy_score(y_true_val, y_pred_val_best)
    acc_test = accuracy_score(y_true_test, y_pred_test_best)
    mcc_test = matthews_corrcoef(y_true_test, y_pred_test_best)

    f1_mac_val = f1_score(y_true_val, y_pred_val_best, average='macro')
    f1_mac_test = f1_score(y_true_test, y_pred_test_best, average='macro')

    f1_wei_val = f1_score(y_true_val, y_pred_val_best, average='weighted')
    f1_wei_test = f1_score(y_true_test, y_pred_test_best, average='weighted')

    recalls_val = recall_score(y_true_val, y_pred_val_best, average=None)
    recalls_test = recall_score(y_true_test, y_pred_test_best, average=None)

    # Imprimir el Reporte en Consola
    print("\n" + "="*70)
    print(f" REPORTE DEL {nombre_modelo.upper()} | MEJOR: {mejor_columna_epoch.upper()}")
    print("="*70)
    print(f"{'Métrica':<18} | {'Validación':<15} | {'Test'}")
    print("-" * 70)
    print(f"{'Accuracy':<18} | {acc_val:.4f}          | {acc_test:.4f}")
    print(f"{'MCC':<18} | {mejor_mcc:.4f}          | {mcc_test:.4f}")
    print(f"{'F1 Macro':<18} | {f1_mac_val:.4f}          | {f1_mac_test:.4f}")
    print(f"{'F1 Weighted':<18} | {f1_wei_val:.4f}          | {f1_wei_test:.4f}")
    print("-" * 70)
    print("Recall por clase:")
    print("-" * 70)

    for clase_num, nombre in mapa_clases.items():
        if clase_num < len(recalls_val):
            r_val = recalls_val[clase_num]
            r_test = recalls_test[clase_num]
            print(f"  • {nombre:<14} | {r_val:.4f} ({r_val*100:.1f}%) | {r_test:.4f} ({r_test*100:.1f}%)")
    print("="*70 + "\n")

    # ==========================================
    # EXPORTACIÓN A CSV
    # ==========================================
    # Limpiamos un poco el nombre del modelo por si le ponemos espacios
    nombre_archivo_seguro = nombre_modelo.replace(" ", "_").lower()

    # --- CSV 1: Métricas Generales ---
    datos_generales = {
        'Metrica': ['Accuracy', 'MCC', 'F1 Macro', 'F1 Weighted'],
        'Validacion': [acc_val, mejor_mcc, f1_mac_val, f1_wei_val],
        'Test': [acc_test, mcc_test, f1_mac_test, f1_wei_test]
    }
    df_generales = pd.DataFrame(datos_generales)
    ruta_csv_general = os.path.join(ruta_salida, f"metricas_generales_{nombre_archivo_seguro}.csv")
    df_generales.to_csv(ruta_csv_general, index=False)

    # --- CSV 2: Recall por Clase ---
    datos_clases = []
    for clase_num, nombre in mapa_clases.items():
        if clase_num < len(recalls_val):
            datos_clases.append({
                'Clase': nombre,
                'Recall_Validacion': recalls_val[clase_num],
                'Recall_Test': recalls_test[clase_num]
            })
    df_clases = pd.DataFrame(datos_clases)
    ruta_csv_clases = os.path.join(ruta_salida, f"recall_por_clase_{nombre_archivo_seguro}.csv")
    df_clases.to_csv(ruta_csv_clases, index=False)

    # Mensaje de confirmación
    print(f" Se han guardado correctamente los resultados de '{nombre_modelo}':")
    print(f"   ➔ {ruta_csv_general}")
    print(f"   ➔ {ruta_csv_clases}\n")

In [ ]:
# ==========================================
# EL BUCLE
# ==========================================
NUMERO_DE_MODELOS = 9
directorio_csvs = '' # Vacío si el script y los csv están juntos

print(" INICIANDO EVALUACIÓN DE MODELOS...")

for i in range(1, NUMERO_DE_MODELOS + 1):
    archivo_val = os.path.join(directorio_csvs, f'val_modelo_{i}.csv')
    archivo_test = os.path.join(directorio_csvs, f'test_modelo_{i}.csv')

    if os.path.exists(archivo_val) and os.path.exists(archivo_test):
        df_validacion = pd.read_csv(archivo_val)
        df_testeo = pd.read_csv(archivo_test)

        # Determinar el Dataset actual y asignarle un nombre
        if i % 3 == 1:
            MAPA_CLASES_ACTUAL = diccionario_dataset_1
            nombre_dataset = "dataset1"
        elif i % 3 == 2:
            MAPA_CLASES_ACTUAL = diccionario_dataset_2
            nombre_dataset = "dataset2"
        elif i % 3 == 0:
            MAPA_CLASES_ACTUAL = diccionario_dataset_3
            nombre_dataset = "dataset3"

        # Determinar la arquitectura y asignarle un nombre
        if i in range(1, 4):
            nombre_arquitectura = "microsoft"
        elif i in range(4, 7):
            nombre_arquitectura = "googleViT"
        elif i in range(7, 10):
            nombre_arquitectura = "facebook_dinov2"

        # CONSTRUIR EL NOMBRE FINAL DESCRIPTIVO
        nombre_descriptivo = f"{nombre_dataset}_{nombre_arquitectura}"

        # =======================================================
        #  CREAR LA CARPETA ORGANIZADA
        # =======================================================
        # Ejemplo: "resultados_modelos/dataset2_googleViT"
        nombre_subcarpeta = f"{nombre_dataset}_{nombre_arquitectura}"
        ruta_guardado = os.path.join("resultados_modelos", nombre_subcarpeta)

        # exist_ok=True evita que de error si la carpeta ya existe de antes.
        os.makedirs(ruta_guardado, exist_ok=True)

        print(f"\n >>> Procesando original modelo {i} <<<")

        # Llamar a la función
        buscar_mejor_epoch_y_reportar(
            df_val=df_validacion,
            df_test=df_testeo,
            nombre_modelo=nombre_descriptivo,
            mapa_clases=MAPA_CLASES_ACTUAL,
            ruta_salida=ruta_guardado
        )
    else:
        print(f" Faltan archivos para el Modelo {i}. Saltando...")

 INICIANDO EVALUACIÓN DE MODELOS...

 >>> Procesando original modelo 1 <<<

 REPORTE DEL DATASET1_MICROSOFT | MEJOR: EPOCA_8
Métrica            | Validación      | Test
----------------------------------------------------------------------
Accuracy           | 0.8000          | 0.7478
MCC                | 0.7222          | 0.6609
F1 Macro           | 0.7992          | 0.6829
F1 Weighted        | 0.7969          | 0.7391
----------------------------------------------------------------------
Recall por clase:
----------------------------------------------------------------------
  • 0_ACK          | 0.8788 (87.9%) | 0.8462 (84.6%)
  • 1_BCC          | 0.8125 (81.2%) | 0.7568 (75.7%)
  • 2_MEL          | 1.0000 (100.0%) | 0.5000 (50.0%)
  • 3_NEV          | 0.6667 (66.7%) | 0.3000 (30.0%)
  • 4_SCC          | 0.5000 (50.0%) | 0.6000 (60.0%)
  • 5_SEK          | 1.0000 (100.0%) | 0.8824 (88.2%)

 Se han guardado correctamente los resultados de 'dataset1_microsoft':
   ➔ resultados_modelos/

In [ ]:
from huggingface_hub import batch_bucket_files
import os
import io
from pathlib import Path

carpeta = Path(f"resultados_modelos")
lista_tuplas = []

# rglob('*') busca recursivamente todos los elementos dentro de la carpeta y subcarpetas
for ruta_archivo in carpeta.rglob('*'):
    # Nos aseguramos de que es un archivo y no una carpeta
    if ruta_archivo.is_file():
        ruta_str = str(ruta_archivo)
        # Añadimos la tupla con la ruta repetida (ruta_local, ruta_destino)
        lista_tuplas.append((ruta_str, ruta_str))

print (lista_tuplas[0], len(lista_tuplas))

api.batch_bucket_files(
    bucket_id,
    add=lista_tuplas,
)

('resultados_modelos/dataset2_googleViT/recall_por_clase_dataset2_googlevit.csv', 'resultados_modelos/dataset2_googleViT/recall_por_clase_dataset2_googlevit.csv') 18


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...es_dataset2_googlevit.csv: 100%|##########|   210B /   210B            

  ...es_dataset2_microsoft.csv: 100%|##########|   210B /   210B            

  ...aset3_facebook_dinov2.csv: 100%|##########|  75.0B /  75.0B            

... (3 outputs left out)

  ...es_dataset1_googlevit.csv: 100%|##########|   209B /   209B            

  ...se_dataset3_microsoft.csv: 100%|##########|  76.0B /  76.0B            

  ...aset3_facebook_dinov2.csv: 100%|##########|   182B /   182B            

  ...aset1_facebook_dinov2.csv: 100%|##########|   210B /   210B            

In [ ]:
from huggingface_hub import list_bucket_tree
from huggingface_hub import download_bucket_files
carpeta_objetivo = "Modelos_Entrenados/experimento_dataset3_facebook_dinov2_SMALL_DESTILADO/"


# Obtenemos la lista exacta de archivos dentro de esa carpeta
archivos_remotos = [
    item.path
    for item in list_bucket_tree(bucket_id, prefix=carpeta_objetivo)
    if item.type == "file"
]
#print(archivos_remotos[0])
# Construimos la lista de tuplas que necesita download_bucket_files
# Formato: (ruta_en_el_bucket, ruta_en_tu_disco_local)
# Al poner (ruta, ruta) se guardarán conservando la misma estructura de carpetas.
archivos_a_descargar = [
    (ruta, ruta)
    for ruta in archivos_remotos
]

print(f"¡Se han encontrado {len(archivos_a_descargar)} archivos para descargar!")

# Descargamos los archivos en bloque
if archivos_a_descargar:
    download_bucket_files(
        bucket_id,
        files=archivos_a_descargar
    )
    print("¡Descarga completada con éxito!")
else:
    print("No se encontró nada para descargar (la carpeta no existe o está vacía).")

¡Se han encontrado 3 archivos para descargar!


¡Descarga completada con éxito!


In [ ]:
import os
import torch
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, matthews_corrcoef, f1_score, recall_score
from transformers import AutoModelForImageClassification # Importación clave

def evaluar_huggingface_desde_ruta(ruta_carpeta, dataloader_val, dataloader_test, nombre_modelo, mapa_clases, ruta_salida, device='cuda'):
    print(f"\n Cargando modelo Hugging Face desde: {ruta_carpeta}")

    # 1. CARGAR EL MODELO
    # Hugging Face lee automáticamente el config.json y el model.safetensors de la carpeta
    modelo = AutoModelForImageClassification.from_pretrained(ruta_carpeta)
    modelo = modelo.to(device)
    modelo.eval()

    # Función interna auxiliar
    def predecir_dataset(dataloader):
        y_true_lista = []
        y_pred_lista = []

        with torch.no_grad():
            for batch in dataloader: # Iteramos recibiendo el diccionario completo
                # Extraemos los tensores del diccionario y los mandamos a la GPU
                pixel_values = batch["pixel_values"].to(device)
                labels = batch["labels"].to(device)

                # Le pasamos solo las imágenes al modelo
                outputs = modelo(pixel_values)
                logits = outputs.logits

                _, preds = torch.max(logits, 1)

                y_true_lista.extend(labels.cpu().numpy())
                y_pred_lista.extend(preds.cpu().numpy())

        return np.array(y_true_lista), np.array(y_pred_lista)
    # 2. GENERAR PREDICCIONES
    print(" ➔ Evaluando Validación...")
    y_true_val, y_pred_val = predecir_dataset(dataloader_val)

    print(" ➔ Evaluando Test...")
    y_true_test, y_pred_test = predecir_dataset(dataloader_test)

    # 3. CÁLCULO DE MÉTRICAS
    acc_val = accuracy_score(y_true_val, y_pred_val)
    acc_test = accuracy_score(y_true_test, y_pred_test)
    mcc_val = matthews_corrcoef(y_true_val, y_pred_val)
    mcc_test = matthews_corrcoef(y_true_test, y_pred_test)
    f1_mac_val = f1_score(y_true_val, y_pred_val, average='macro')
    f1_mac_test = f1_score(y_true_test, y_pred_test, average='macro')
    f1_wei_val = f1_score(y_true_val, y_pred_val, average='weighted')
    f1_wei_test = f1_score(y_true_test, y_pred_test, average='weighted')
    recalls_val = recall_score(y_true_val, y_pred_val, average=None)
    recalls_test = recall_score(y_true_test, y_pred_test, average=None)

    # 4. EXPORTAR A CSV (Creando la carpeta de resultados)
    nombre_archivo_seguro = nombre_modelo.replace(" ", "_").lower()
    os.makedirs(ruta_salida, exist_ok=True)

    datos_generales = {
        'Metrica': ['Accuracy', 'MCC', 'F1 Macro', 'F1 Weighted'],
        'Validacion': [acc_val, mcc_val, f1_mac_val, f1_wei_val],
        'Test': [acc_test, mcc_test, f1_mac_test, f1_wei_test]
    }
    pd.DataFrame(datos_generales).to_csv(os.path.join(ruta_salida, f"metricas_generales_{nombre_archivo_seguro}.csv"), index=False)

    datos_clases = []
    for clase_num, nombre in mapa_clases.items():
        if clase_num < len(recalls_val):
            datos_clases.append({
                'Clase': nombre,
                'Recall_Validacion': recalls_val[clase_num],
                'Recall_Test': recalls_test[clase_num]
            })
    pd.DataFrame(datos_clases).to_csv(os.path.join(ruta_salida, f"recall_por_clase_{nombre_archivo_seguro}.csv"), index=False)

    print(f" Guardado con éxito en: {ruta_salida}\n")

In [ ]:
from datasets import load_dataset
dataset = load_dataset("z72pepee/skin-gan-review-baseline", "PH2", token="mi_token")

README.md:   0%|          | 0.00/35.4k [00:00<?, ?B/s]

PH2/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 11.6MB            

PH2/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

PH2/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.44MB            

PH2/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

PH2/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.49MB            

PH2/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/160 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/20 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/20 [00:00<?, ? examples/s]

In [ ]:
import torch
from torch.utils.data import DataLoader
from transformers import AutoModelForImageClassification, AutoImageProcessor, default_data_collator
from transformers import AutoImageProcessor


ruta_local = "Modelos_Entrenados/experimento_dataset3_facebook_dinov2_SMALL_DESTILADO/"
processor = AutoImageProcessor.from_pretrained(ruta_local)

# Definir la función de transformación
def transform_images(examples):
    images = [img.convert("RGB") for img in examples["image"]]
    inputs = processor(images, return_tensors="pt")
    inputs["labels"] = examples["label"]
    return inputs

# Aplicar la transformación

processed_val_dataset = dataset["validation"].with_transform(transform_images)
processed_test_dataset = dataset["test"].with_transform(transform_images)

# Crear DataLoader
test_loader = DataLoader(
    processed_test_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=default_data_collator
)


val_loader = DataLoader(
    processed_val_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=default_data_collator
)

In [ ]:
ruta_carpeta_experimento = "Modelos_Entrenados/experimento_dataset3_facebook_dinov2_SMALL_DESTILADO/"

evaluar_huggingface_desde_ruta(
    ruta_carpeta=ruta_carpeta_experimento,
    dataloader_val=val_loader,
    dataloader_test=test_loader,
    nombre_modelo="dataset3_facebook_dinov2_small_destilado",
    mapa_clases=diccionario_dataset_3,
    ruta_salida="resultados_modelos/dataset3_facebook_dinov2_small_destilado"
)


 Cargando modelo Hugging Face desde: Modelos_Entrenados/experimento_dataset3_facebook_dinov2_SMALL_DESTILADO/


Loading weights:   0%|          | 0/225 [00:00<?, ?it/s]

 ➔ Evaluando Validación...
 ➔ Evaluando Test...
 Guardado con éxito en: resultados_modelos/dataset3_facebook_dinov2_small_destilado



In [ ]:
from huggingface_hub import batch_bucket_files
import os
import io
from pathlib import Path

carpeta = Path(f"resultados_modelos")
lista_tuplas = []

# rglob('*') busca recursivamente todos los elementos dentro de la carpeta y subcarpetas
for ruta_archivo in carpeta.rglob('*'):
    # Nos aseguramos de que es un archivo y no una carpeta
    if ruta_archivo.is_file():
        ruta_str = str(ruta_archivo)
        # Añadimos la tupla con la ruta repetida (ruta_local, ruta_destino)
        lista_tuplas.append((ruta_str, ruta_str))

print (lista_tuplas[0], len(lista_tuplas))

api.batch_bucket_files(
    bucket_id,
    add=lista_tuplas,
)

('resultados_modelos/dataset3_facebook_dinov2_small_destilado/recall_por_clase_dataset3_facebook_dinov2_small_destilado.csv', 'resultados_modelos/dataset3_facebook_dinov2_small_destilado/recall_por_clase_dataset3_facebook_dinov2_small_destilado.csv') 2


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...inov2_small_destilado.csv: 100%|##########|   136B /   136B            

  ...inov2_small_destilado.csv: 100%|##########|  72.0B /  72.0B            